# Federal urgency-contract tracker (FAR 6.302-2)

When the government skips competition by invoking **unusual and compelling urgency** — the standard that it would suffer "serious injury, financial or other," without it — it records that in the federal contract data as the *other than full and open competition* reason **`URGENCY (FAR 6.302-2)`**.

This notebook tracks **how much that exception gets used over time, and where the money goes** — straight from the [`abigailhaddad/usaspending-bulk-awards`](https://huggingface.co/datasets/abigailhaddad/usaspending-bulk-awards) mirror of USAspending, queried with DuckDB over `hf://`. **No download, no API key, no rate limit.**

The headline: FY2026 is already at ~$28B of urgency-justified contracts — and **88% of it is border-wall construction at Customs and Border Protection.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/urgency-tracker/blob/main/demo.ipynb)

In [ ]:
%pip install -q duckdb pandas matplotlib

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# One parquet per fiscal year in the serving layer — read by direct URL (no recursive
# listing, which HuggingFace rate-limits).
SERVE = "https://huggingface.co/datasets/abigailhaddad/usaspending-bulk-awards/resolve/main/serve/contracts"
def year_file(fy):
    return f"read_parquet('{SERVE}/{fy}.parquet')"

# FPDS records the urgency exception in this field.
URGENCY = "other_than_full_and_open_competition ILIKE '%URGENCY%'"

## 1. Urgency usage over time

In [ ]:
rows = []
for fy in range(2015, 2027):
    r = con.execute(f"""
      SELECT
        count(*)                  FILTER (WHERE {URGENCY}) AS urgency_actions,
        sum(federal_action_obligation) FILTER (WHERE {URGENCY}) AS urgency_dollars,
        sum(federal_action_obligation) FILTER (WHERE other_than_full_and_open_competition IS NOT NULL)
                                                                 AS all_noncompete_dollars
      FROM {year_file(fy)}
    """).fetchone()
    rows.append({"fiscal_year": fy, "urgency_actions": r[0] or 0,
                 "urgency_dollars": float(r[1] or 0), "all_noncompete_dollars": float(r[2] or 0)})

trend = pd.DataFrame(rows)
trend["urgency_share"] = trend["urgency_dollars"] / trend["all_noncompete_dollars"]
trend.assign(
    urgency_dollars=lambda d: (d.urgency_dollars / 1e9).round(2).astype(str) + "B",
    urgency_share=lambda d: (d.urgency_share * 100).round(1).astype(str) + "%",
)[["fiscal_year", "urgency_actions", "urgency_dollars", "urgency_share"]]

## 2. The trend chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
plt.rcParams["text.parse_math"] = False
ACCENT = "#69539E"

fys = trend["fiscal_year"].tolist()
dollars = (trend["urgency_dollars"] / 1e9).tolist()
partial = [fy == 2026 for fy in fys]

fig, ax = plt.subplots(figsize=(11, 6))
fig.subplots_adjust(top=0.82, bottom=0.10, left=0.08, right=0.95)
ax.bar(range(len(fys)), dollars,
       color=[ACCENT if not p else "#b9a8d6" for p in partial],
       hatch=["" if not p else "//" for p in partial])
for i, d in enumerate(dollars):
    ax.text(i, d + max(dollars) * 0.02, f"${d:,.0f}B", ha="center", va="bottom",
            fontsize=9, fontweight="bold", color="#222")

ax.set_xticks(range(len(fys)))
ax.set_xticklabels([f"FY{f}" + ("*" if p else "") for f, p in zip(fys, partial)], fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}B"))
ax.set_ylim(0, max(dollars) * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#ededed", lw=0.8); ax.set_axisbelow(True)

fig.text(0.08, 0.95, "Federal contracting under the urgency exception (FAR 6.302-2)",
         fontsize=15, fontweight="bold", color=ACCENT, ha="left", va="top")
fig.text(0.08, 0.89,
         "Dollars obligated under “unusual and compelling urgency.”  *FY2026 is a partial year.",
         fontsize=9.5, color="#555", ha="left", va="top")
fig.text(0.08, 0.015,
         "Source: USAspending bulk award archive (U.S. Government, public domain), mirrored at "
         "huggingface.co/datasets/abigailhaddad/usaspending-bulk-awards.",
         fontsize=7.5, color="#999", ha="left", va="bottom")
fig.savefig("urgency_trend.png", dpi=200, facecolor="white")
plt.show()

## 3. Who uses it — by agency

In [ ]:
con.execute(f"""
  SELECT awarding_agency_name AS agency,
         count(*) AS actions,
         round(sum(federal_action_obligation) / 1e6, 1) AS dollars_millions
  FROM {year_file(2026)}
  WHERE {URGENCY}
  GROUP BY 1
  ORDER BY dollars_millions DESC
  LIMIT 12
""").df()

## 4. The top urgency contracts (FY2026)

The biggest urgency-justified awards. In FY2026 these are almost entirely **border-wall construction at CBP** — the descriptions say so outright.

In [ ]:
con.execute(f"""
  SELECT award_id_piid AS piid,
         any_value(recipient_name) AS recipient,
         any_value(awarding_sub_agency_name) AS sub_agency,
         round(sum(federal_action_obligation) / 1e6, 1) AS obligated_millions,
         any_value(prime_award_base_transaction_description) AS description
  FROM {year_file(2026)}
  WHERE {URGENCY}
  GROUP BY 1
  ORDER BY obligated_millions DESC
  LIMIT 15
""").df()

## 5. The full granular data — every urgency contract

The complete award-level table, saved to CSV so you can open it anywhere. The repo ships `urgency_contracts_fy2026.csv` and a one-command script (`python urgency_contracts.py --year 2026`).

In [ ]:
# The full award-level table — every FY2026 urgency contract.
# (This is exactly what urgency_contracts.py writes to CSV.)
urgency_2026 = con.execute(f"""
  SELECT award_id_piid                                  AS piid,
         any_value(recipient_name)                      AS recipient,
         any_value(awarding_sub_agency_name)            AS sub_agency,
         round(sum(federal_action_obligation), 2)       AS obligated,
         count(*)                                       AS actions,
         min(action_date)                               AS first_action,
         max(action_date)                               AS last_action,
         any_value(product_or_service_code_description) AS psc_description,
         any_value(prime_award_base_transaction_description) AS description
  FROM {year_file(2026)}
  WHERE {URGENCY}
  GROUP BY 1
  ORDER BY obligated DESC
""").df()

urgency_2026.to_csv("urgency_contracts_fy2026.csv", index=False)
print(f"{len(urgency_2026):,} FY2026 urgency awards, "
      f"${urgency_2026.obligated.sum()/1e9:.1f}B obligated -> urgency_contracts_fy2026.csv")
urgency_2026.head(15)

## Sources & caveats

- **Source:** USAspending bulk award archive (FPDS prime contracts), a U.S. Government public-domain work, mirrored at [`abigailhaddad/usaspending-bulk-awards`](https://huggingface.co/datasets/abigailhaddad/usaspending-bulk-awards) and queried with DuckDB over `hf://`. No key, no rate limit.
- **Complete, not a sample:** competition coding is mandatory FPDS reporting on every contract action, so these numbers are the full universe — not a search-API slice.
- **Transaction-level:** rows are contract *actions* (modifications); dollars are `federal_action_obligation` summed (can include de-obligations). An "award" is grouped by `award_id_piid`.
- **FY2026 is partial** — through the latest archive snapshot — so action counts are low even where dollars are high.
- **Urgency** = the `other_than_full_and_open_competition` field containing `URGENCY (FAR 6.302-2)`. This is the agency’s own coding; USAspending records the reason *code*, not the written justification narrative.